In [24]:
import pandas as pd
from pathlib import Path
import re

In [25]:
base_dir = Path(r"D:\Projects\Community-Pulse\data")
yearly_root = base_dir / "yearly-data"
metadata_root = base_dir / "metadata"
output_root = base_dir / "combined-data"

In [26]:
def find_metadata_file(folder_name):
    metadata_file = metadata_root / f"{folder_name}-metadata.csv"
    if metadata_file.exists():
        return metadata_file
    return None

def read_metadata(metadata_file):
    meta = pd.read_csv(metadata_file, dtype=str)
    meta.columns = meta.columns.str.strip()

    meta = meta[["Column Name", "Label"]].copy()
    meta["Column Name"] = meta["Column Name"].str.strip()
    meta["Label"] = meta["Label"].str.strip()

    return meta

In [27]:
def make_unique(names):
    counts = {}
    result = []

    for name in names:
        if name not in counts:
            counts[name] = 0
            result.append(name)
        else:
            counts[name] += 1
            result.append(f"{name}_{counts[name]}")

    return result

In [28]:
def process_data_file(file_path, year, meta):
    raw = pd.read_csv(file_path, header=None, dtype=str, keep_default_na=False)

    # row 1 in original file = codes
    file_codes = raw.iloc[0].tolist()

    # keep only nonblank file columns
    keep_idx = [i for i, c in enumerate(file_codes) if str(c).strip() != ""]
    raw = raw.iloc[:, keep_idx]
    file_codes = [file_codes[i] for i in keep_idx]

    # actual data starts after first 2 rows
    data_only = raw.iloc[2:].copy()
    data_only.columns = file_codes

    # master metadata order
    meta_codes = meta["Column Name"].tolist()
    meta_labels = meta["Label"].tolist()

    # align to full metadata structure
    aligned = pd.DataFrame(index=data_only.index)

    for code in meta_codes:
        if code in data_only.columns:
            aligned[code] = data_only[code]
        else:
            aligned[code] = pd.NA

    # rename columns to labels in metadata order
    final_labels = make_unique(meta_labels)
    aligned.columns = final_labels

    # add Year as first column
    aligned.insert(0, "Year", year)

    return aligned

In [29]:
for folder_path in yearly_root.iterdir():
    if not folder_path.is_dir():
        continue

    folder_name = folder_path.name.lower()
    metadata_file = find_metadata_file(folder_name)

    if metadata_file is None:
        print(f"Skipped {folder_name}: metadata file not found")
        continue

    print(f"\nProcessing folder: {folder_name}")
    print(f"Using metadata: {metadata_file.name}")

    meta = read_metadata(metadata_file)
    frames = []

    for file_path in sorted(folder_path.glob("*-Data.csv")):
        match = re.search(r"(2019|2020|2021|2022|2023)", file_path.name)
        if not match:
            print(f"  Skipped {file_path.name}: year not found")
            continue

        year = int(match.group(1))

        try:
            df_year = process_data_file(file_path, year, meta)
            frames.append(df_year)
            print(f"  Added {file_path.name} -> shape {df_year.shape}")
        except Exception as e:
            print(f"  Error in {file_path.name}: {e}")

    if frames:
        combined = pd.concat(frames, ignore_index=True, sort=False)
        output_file = output_root / f"{folder_name}_combined.csv"
        combined.to_csv(output_file, index=False)
        print(f"Saved: {output_file.name}, shape={combined.shape}")
    else:
        print(f"No valid files found for {folder_name}")

print("\nDone")


Processing folder: age
Using metadata: age-metadata.csv
  Added age2019-Data.csv -> shape (29, 459)
  Added age2020-Data.csv -> shape (30, 459)
  Added age2021-Data.csv -> shape (30, 459)
  Added age2022-Data.csv -> shape (30, 459)
  Added age2023-Data.csv -> shape (30, 459)
Saved: age_combined.csv, shape=(149, 459)

Processing folder: economiccharateristic
Using metadata: economiccharateristic-metadata.csv
  Added economiccharateristic2019-Data.csv -> shape (27, 551)
  Added economiccharateristic2020-Data.csv -> shape (28, 551)
  Added economiccharateristic2021-Data.csv -> shape (28, 551)
  Added economiccharateristic2022-Data.csv -> shape (28, 551)
  Added economiccharateristic2023-Data.csv -> shape (28, 551)
Saved: economiccharateristic_combined.csv, shape=(139, 551)

Processing folder: employmentstatus
Using metadata: employmentstatus-metadata.csv
  Added employmentstatus2019-Data.csv -> shape (27, 283)
  Added employmentstatus2020-Data.csv -> shape (28, 283)
  Added employmentsta